# 05. Точечные снимки технологий (метод 18)

Снимки — числа на **конкретную дату / период** из внешнего отчёта. Это не полный ряд 2010–2023 и не тест гипотез H1–H6.

Источники:
- Stanford HAI. *AI Index Report 2025*.
- TOP500 List (ноябрь 2015 и ноябрь 2025).
- EPO–OECD. *Mapping the global quantum ecosystem* (2025).

Подробная сверка: `SNAPSHOT_SOURCE_LEDGER.md`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
PANEL_PATH = ROOT / "data_reviewed" / "core_panel_reviewed.csv"
TABLES = ROOT / "data_reviewed" / "tables_reviewed"
RAW = ROOT / "data" / "raw"
OUT_FIG = ROOT / "notebooks" / "figures"
OUT_TAB = ROOT / "notebooks" / "tables"
OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TAB.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PANEL_PATH)
panel = panel[(panel["year"] >= 2010) & (panel["year"] <= 2024)].copy()
US_CN = panel[panel["country_iso3"].isin(["USA", "CHN"])].copy()

def cagr(start: float, end: float, n_years: int) -> float:
    """Compound annual growth rate; n_years = end_year - start_year."""
    if start is None or end is None or np.isnan(start) or np.isnan(end) or start <= 0 or n_years <= 0:
        return np.nan
    return (end / start) ** (1 / n_years) - 1

print("panel rows", len(panel), "US+CN rows", len(US_CN))
print("countries", sorted(panel["country_iso3"].unique()))


panel rows 120 US+CN rows 30
countries ['CHN', 'DEU', 'FRA', 'GBR', 'ISR', 'JPN', 'KOR', 'USA']


In [2]:
ai = pd.read_csv(RAW / "snapshots" / "ai_index_snapshot.csv")
top = pd.read_csv(RAW / "snapshots" / "top500_snapshot.csv")
quantum = pd.read_csv(RAW / "snapshots" / "quantum_epo_oecd_snapshot.csv")
ai.to_csv(OUT_TAB / "05_ai_index_snapshot.csv", index=False)
top.to_csv(OUT_TAB / "05_top500_snapshot.csv", index=False)
quantum.to_csv(OUT_TAB / "05_quantum_snapshot.csv", index=False)


## Искусственный интеллект (Stanford AI Index)


In [3]:
ai_us_cn = ai[ai["country_iso3"].isin(["USA", "CHN"])].copy()
ai_us_cn


,country_iso3,year,indicator,value,unit,source,page_or_figure,notes
0,CHN,2023,ai_publications_share_of_world,23.2,percent of AI CS pubs,Stanford HAI AI Index Report 2025 Chapter 1,Figure 1.1.6 / highlight p.2,English-language AI pubs in OpenAlex CS; volum...
1,USA,2023,ai_publications_share_of_world,9.2,percent of AI CS pubs,Stanford HAI AI Index Report 2025 Chapter 1,Figure 1.1.6,Same definition; USA below China on volume share
2,CHN,2023,ai_citations_share_of_world,22.6,percent of AI CS citations,Stanford HAI AI Index Report 2025 Chapter 1,Figure 1.1.7 / highlight,Citation share; still volume-adjacent; USA lea...
3,USA,2023,ai_citations_share_of_world,13.0,percent of AI CS citations,Stanford HAI AI Index Report 2025 Chapter 1,Figure 1.1.7,Citation share
4,USA,2024,ai_private_investment_usd_bn,109.1,USD billion,Stanford HAI AI Index Report 2025 Economy chap...,Highlight 3 / Economy ch.,Private investment; CN opaque risk remains; no...
5,CHN,2024,ai_private_investment_usd_bn,9.3,USD billion,Stanford HAI AI Index Report 2025 Economy chap...,Highlight 3 / Economy ch.,Private investment; ~12x USA/China gap in repo...
6,USA,2024,ai_notable_models_count,40.0,count,Stanford HAI AI Index Report 2025 Chapter 1,Figure 1.3.1 / highlight 4,Institution affiliation of notable models; COM...
7,CHN,2024,ai_notable_models_count,15.0,count,Stanford HAI AI Index Report 2025 Chapter 1,Figure 1.3.1 / highlight 4,Same definition; not deployment/adoption
8,CHN,2023,ai_patents_share_of_world_grants,69.7,percent of AI patent grants,Stanford HAI AI Index Report 2025 Chapter 1,Figure 1.2.3,Patent grants share; quantity not quality
9,USA,2023,ai_patents_share_of_world_grants,14.2,percent of AI patent grants,Stanford HAI AI Index Report 2025 Chapter 1,Figure 1.2.3,Patent grants share


In [4]:
# Simple grouped bars for selected AI indicators
focus = [
    "ai_publications_share_of_world",
    "ai_citations_share_of_world",
    "ai_private_investment_usd_bn",
    "ai_notable_models_count",
    "ai_patents_share_of_world_grants",
]
plot_df = ai_us_cn[ai_us_cn["indicator"].isin(focus)].copy()
fig, axes = plt.subplots(1, len(focus), figsize=(14, 3.5), sharey=False)
for ax, ind in zip(axes, focus):
    sub = plot_df[plot_df["indicator"] == ind]
    order = ["USA", "CHN"]
    vals = [sub.loc[sub["country_iso3"] == c, "value"].values[0] for c in order]
    ax.bar(order, vals, color=["#1f77b4", "#d62728"])
    ax.set_title(ind.replace("_", "\n"), fontsize=8)
    ax.grid(True, axis="y", alpha=0.3)
fig.suptitle("Снимки AI Index: США vs Китай (годы см. таблицу)", y=1.05)
fig.tight_layout()
fig.savefig(OUT_FIG / "05_ai_index_bars.png", dpi=150, bbox_inches="tight")
plt.show()


C:\Users\danii\AppData\Local\Temp\ipykernel_22864\2223774712.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Суперкомпьютеры (TOP500)


In [5]:
top_us_cn = top[top["country_iso3"].isin(["USA", "CHN"])].copy()
pivot = top_us_cn.pivot_table(index="list_month", columns="country_iso3", values="value")
print(top_us_cn.to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 4))
pivot[["USA", "CHN"]].plot(kind="bar", ax=ax, color=["#1f77b4", "#d62728"])
ax.set_title("Число систем в TOP500")
ax.set_ylabel("Системы")
ax.set_xlabel("Список")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_FIG / "05_top500_counts.png", dpi=150)
plt.show()
top_us_cn


country_iso3  year list_month            indicator  value  unit                                                                                 source                                                                                                      page_or_figure                                                   notes
         USA  2015    2015-11 top500_systems_count    199 count                                                   TOP500 List Highlights November 2015                                                             https://new.top500.org/lists/top500/2015/11/highlights/       HPC Linpack systems on list; not cloud AI compute
         CHN  2015    2015-11 top500_systems_count    109 count                                                   TOP500 List Highlights November 2015                                                             https://new.top500.org/lists/top500/2015/11/highlights/       HPC Linpack systems on list; not cloud AI compute
         USA  2025    2025-11 t

C:\Users\danii\AppData\Local\Temp\ipykernel_22864\1337936261.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,country_iso3,year,list_month,indicator,value,unit,source,page_or_figure,notes
0,USA,2015,2015-11,top500_systems_count,199,count,TOP500 List Highlights November 2015,https://new.top500.org/lists/top500/2015/11/hi...,HPC Linpack systems on list; not cloud AI compute
1,CHN,2015,2015-11,top500_systems_count,109,count,TOP500 List Highlights November 2015,https://new.top500.org/lists/top500/2015/11/hi...,HPC Linpack systems on list; not cloud AI compute
2,USA,2025,2025-11,top500_systems_count,171,count,TOP500 / Network World summary of Nov 2025 lis...,https://www.networkworld.com/article/4096893/e...,Nov 2025; USA down 2 vs June; not cloud AI com...
3,CHN,2025,2025-11,top500_systems_count,40,count,TOP500 / Network World summary of Nov 2025 lis...,https://www.networkworld.com/article/4096893/e...,Nov 2025; China down 6 vs June; tied with Germ...


## Квантовые технологии (EPO–OECD, международные патентные семьи)


In [6]:
q = quantum.copy()
q_us_cn = q[q["country_iso3"].isin(["USA", "CHN", "JPN", "WORLD"])]
q_us_cn


,country_iso3,year_span,indicator,value,unit,source,page_or_figure,notes
0,USA,2005-2024,quantum_ipf_count,3330,international patent families,EPO-OECD Mapping the global quantum ecosystem ...,Figure 3.3.1 Panel B text,IPFs only; China leads national-only families ...
1,CHN,2005-2024,quantum_ipf_count,947,international patent families,EPO-OECD Mapping the global quantum ecosystem ...,Figure 3.3.1 Panel B text,Fourth among countries after USA EPC Japan; na...
2,JPN,2005-2024,quantum_ipf_count,1519,international patent families,EPO-OECD Mapping the global quantum ecosystem ...,Figure 3.3.1 Panel B text,Comparator
3,USA,2015-2019,quantum_ipf_world_share,41,percent of quantum IPFs,EPO-OECD Mapping the global quantum ecosystem ...,Figure E4 / executive summary,Share declined later
4,USA,2020-2024,quantum_ipf_world_share,31,percent of quantum IPFs,EPO-OECD Mapping the global quantum ecosystem ...,Figure E4 / executive summary,Still largest national share in period
5,WORLD,2005-2024,quantum_ipf_count_total,9740,international patent families,EPO press release citing EPO-OECD study 17 Dec...,https://www.epo.org/en/news-events/press-centr...,Global total for context
6,USA,ever_recorded,quantum_funding_share_to_firms,60,percent of recorded quantum funding,EPO-OECD Mapping the global quantum ecosystem ...,Figure E6 / exec summary,~60% funding to US-based companies; ~30% of IP...
7,CHN,2022,quantum_publications_fractional_count_label,2592,publications (figure label),EPO-OECD report Figure 8.2.7 bubble labels,Figure 8.2.7 p.123 area,Exploratory label from figure; EU/US/CN lead q...
8,USA,2022,quantum_publications_fractional_count_label,1806,publications (figure label),EPO-OECD report Figure 8.2.7 bubble labels,Figure 8.2.7 p.123 area,Exploratory; pair with CN label; commercialisa...


## Оговорки по полупроводникам и Тайваню (½ страницы максимум)

- Код **HS8542** в торговой статистике — экспорт электронных интегральных схем по таможенной классификации. Это **не** измерение мощности фабрик (fab capacity) и не «лидерство в полупроводниках».
- **Тайвань** в панели страны **нет**. Роль foundry на Тайване — качественная оговорка (`qual`), не расчёт. Франция в панели **не** заменяет Тайвань как foundry.

Матрица заполненности 8 блоков × 4 технологии — справочно, не кульминация: см. `tech_occupancy_matrix.md`.


In [7]:
occ = pd.read_csv(ROOT / "tech_occupancy_matrix.csv")
occ.to_csv(OUT_TAB / "05_occupancy_matrix.csv", index=False)
occ


,tci_stage,tci_name,AI,Semiconductors,HPC,Quantum,notes
0,S,science,snapshot: AI pubs share 2023 (AI Index Fig 1.1.6),?,?,snapshot: quantum pubs labels 2022 (EPO-OECD F...,snapshot ≠ measured; general articles NOT doma...
1,HC,human_capital,?,?,?,?,tech-specific HC missing
2,RD,R&D,?,?,?,?,GERD/BERD are general-macro only
3,FIN,finance,snapshot: private AI USD bn 2024 (AI Index),?,?,snapshot: quantum funding share US firms ~60% ...,FIN mix still unmeasured; BERD=PERFORMED
4,INN,innovation,snapshot: AI patent grant share 2023 + notable...,?,?,snapshot: quantum IPF USA 3330 / CHN 947 (2005...,field patents for semis not collected
5,COM,commercialization,snapshot: notable models 2024 (thin),?,?,?,adoption/revenue not measured
6,PRD,production_scaling,?,qual: TWN foundry hole,snapshot: TOP500 systems Nov2015 & Nov2025,?,HS8542 must NOT fill PRD; TOP500 ≠ cloud AI co...
7,ADE,adoption_export,?,measured: semi_exports_hs8542 + semi_status (≠...,?,?,Only measured tech cell; trade ≠ fab/node


## Краткий вывод для отчёта (метод 18)

Снимки показывают **разные знаки** по разным индикаторам (например, Китай выше по доле AI-публикаций и патентов; США выше по частным инвестициям в ИИ и по числу notable models; США выше по международным патентным семьям в кванте; число систем TOP500 у США выше в обоих выбранных моментах). Это точечные факты из источников, а не ряд и не объявление победителя.
